# Remote Spectrum Bucket Spectrograms

This notebook runs in Colab and plots unsmoothed spectrograms directly from the object-store bucket. It expects collector output under prefixes like `s3://spectrum/ara/feedmill-ue-001/<run>/<band>/power_1mhz_avg_per_minute.csv`.


In [ ]:
!pip -q install fsspec s3fs


In [ ]:
import io
import json
import os
from pathlib import PurePosixPath

import fsspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


## Configure Access

Set credentials in Colab secrets or paste them into environment variables for the session. Do not save real keys in the notebook.


In [ ]:
# Option A: set these manually for this Colab session.
# os.environ['AWS_ACCESS_KEY_ID'] = '...'
# os.environ['AWS_SECRET_ACCESS_KEY'] = '...'

S3_ENDPOINT_URL = os.environ.get('S3_ENDPOINT_URL', 'https://chi.tacc.chameleoncloud.org:7480')
BUCKET = os.environ.get('SPECTRUM_BUCKET', 'spectrum')
ROOT_PREFIX = os.environ.get('SPECTRUM_PREFIX', '')

fs_kwargs = {
    'skip_instance_cache': True,
    'use_listings_cache': False,
}
if S3_ENDPOINT_URL:
    fs_kwargs['client_kwargs'] = {'endpoint_url': S3_ENDPOINT_URL}
    fs_kwargs['config_kwargs'] = {'s3': {'addressing_style': 'path'}}

fs = fsspec.filesystem('s3', **fs_kwargs)
bucket_root = f'{BUCKET}/{ROOT_PREFIX}'.rstrip('/')

print('endpoint:', S3_ENDPOINT_URL if S3_ENDPOINT_URL else '(default)')
print('bucket_root:', f's3://{bucket_root}')


## Discover CSVs

The scanner finds every `power_1mhz_avg_per_minute.csv` under the bucket root and parses `site`, `node`, `run`, and `band` from the object key.


In [ ]:
def discover_power_csvs(fs, bucket_root):
    pattern = f'{bucket_root}/**/power_1mhz_avg_per_minute.csv'
    paths = sorted(fs.glob(pattern))
    rows = []
    for path in paths:
        rel = path[len(bucket_root):].lstrip('/')
        parts = rel.split('/')
        if len(parts) < 5:
            continue
        site, node = parts[0], parts[1]
        run = '/'.join(parts[2:-2])
        band = parts[-2]
        rows.append({
            'site': site,
            'node': node,
            'run': run,
            'band': band,
            'csv_path': path,
            'metadata_path': path.rsplit('/', 1)[0] + '/metadata.json',
        })
    return pd.DataFrame(rows)

index = discover_power_csvs(fs, bucket_root)
print('csv_count:', len(index))
index


## Available Data

This summarizes what is currently in object storage. Start time is estimated from the remote CSV update time and the number of one-minute rows.


In [ ]:
def remote_modified_utc(fs, path):
    try:
        info = fs.info(path)
    except FileNotFoundError:
        return pd.NaT
    for key in ('LastModified', 'last_modified', 'mtime', 'updated'):
        if key in info and info[key] is not None:
            return pd.to_datetime(info[key], utc=True, errors='coerce')
    return pd.NaT

def csv_row_count(fs, path):
    with fs.open(path, 'rb') as f:
        return len(pd.read_csv(f))

def summarize_available(index, fs):
    records = []
    for _, row in index.iterrows():
        rows = csv_row_count(fs, row['csv_path'])
        modified = remote_modified_utc(fs, row['csv_path'])
        approx_start = modified - pd.Timedelta(minutes=max(rows - 1, 0)) if pd.notna(modified) else pd.NaT
        records.append({
            'site': row['site'],
            'node': row['node'],
            'run': row['run'],
            'band': row['band'],
            'rows': rows,
            'approx_start_utc': approx_start,
            'most_recent_upload_utc': modified,
        })

    detail = pd.DataFrame(records)
    if detail.empty:
        return detail, detail

    summary = (
        detail.sort_values(['site', 'node', 'run', 'band'])
        .groupby(['site', 'node', 'run'], as_index=False)
        .agg(
            bands=('band', lambda x: ', '.join(x)),
            rows_per_band=('rows', lambda x: ', '.join(str(v) for v in x)),
            approx_start_utc=('approx_start_utc', 'min'),
            most_recent_upload_utc=('most_recent_upload_utc', 'max'),
        )
    )
    return detail, summary

availability_detail, availability_summary = summarize_available(index, fs)
availability_summary


## Optional Filters

Leave lists empty to plot everything found. `MAX_TIME = 0` plots the full CSV.


In [ ]:
SITES = []       # e.g. ['ara', 'powder', 'cosmos']
NODES = []       # e.g. ['feedmill-ue-001']
RUNS = []        # e.g. ['ara_500_700_2400_2600_3500_3700']
BANDS = []       # e.g. ['500_700', '2400_2600', '3500_3700']
MAX_TIME = 0     # 0 means all rows
FROM_END = False

def apply_filters(index):
    df = index.copy()
    if SITES:
        df = df[df['site'].isin(SITES)]
    if NODES:
        df = df[df['node'].isin(NODES)]
    if RUNS:
        df = df[df['run'].isin(RUNS)]
    if BANDS:
        df = df[df['band'].isin(BANDS)]
    return df.reset_index(drop=True)

selected = apply_filters(index)
print('selected_count:', len(selected))
selected


## Plot Helpers

These helpers follow `plot_band_ground_truth.py`: serif fonts, `viridis`, percentile color limits, nearest-neighbor pixels, MHz y-axis, and no smoothing.


In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'serif'],
    'axes.linewidth': 1.0,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
})

def read_json_if_exists(fs, path):
    try:
        with fs.open(path, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        return {}

def read_remote_band(row):
    with fs.open(row['csv_path'], 'rb') as f:
        df = pd.read_csv(f)
    freqs_mhz = df.columns.to_numpy(dtype=float)
    power = df.to_numpy(dtype=float)
    metadata = read_json_if_exists(fs, row['metadata_path'])
    return freqs_mhz, power, metadata

def title_for(row, metadata):
    start_mhz = metadata.get('frequency_start_mhz')
    stop_mhz = metadata.get('frequency_stop_mhz')
    band = f'{start_mhz}-{stop_mhz} MHz' if start_mhz is not None and stop_mhz is not None else row['band'].replace('_', '-') + ' MHz'
    label = f"{row['site']} / {row['node']} / {row['run']}"
    return f'{label} {band}\nGround-Truth'

def trim_time(power, max_time=0, from_end=False):
    if max_time and power.shape[0] > max_time:
        return power[-max_time:] if from_end else power[:max_time]
    return power

def plot_band(row, max_time=0, from_end=False, save=False, output_dir='spectrograms'):
    freqs_mhz, power, metadata = read_remote_band(row)
    if power.shape[1] != 200:
        raise ValueError(f"expected 200 columns in {row['csv_path']}, found {power.shape[1]}")
    power = trim_time(power, max_time=max_time, from_end=from_end)

    finite = power[np.isfinite(power)]
    if finite.size == 0:
        raise ValueError(f"no finite values in {row['csv_path']}")
    vmin, vmax = np.percentile(finite, [1, 99])

    fig, ax = plt.subplots(figsize=(5.2, 1.8))
    im = ax.imshow(
        power.T,
        origin='lower',
        aspect='auto',
        cmap='viridis',
        vmin=vmin,
        vmax=vmax,
        interpolation='nearest',
        extent=[0, power.shape[0], freqs_mhz[0] - 0.5, freqs_mhz[-1] + 0.5],
    )

    start_tick = int(np.ceil((freqs_mhz[0] - 0.5) / 25.0) * 25)
    stop_tick = int(np.floor((freqs_mhz[-1] + 0.5) / 25.0) * 25)
    yticks = np.arange(start_tick, stop_tick + 1, 25)

    ax.set_title(title_for(row, metadata), fontsize=7.5, fontweight='bold', pad=2)
    ax.set_xlabel('Time Step', fontsize=7.5, fontweight='bold', labelpad=1)
    ax.set_ylabel('Frequency (MHz)', fontsize=7.5, fontweight='bold', labelpad=1)
    ax.set_xlim(0, power.shape[0])
    ax.set_yticks(yticks)
    ax.tick_params(labelsize=6.5, top=True, right=True, length=3, width=0.9, pad=1)

    colorbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.04)
    colorbar.ax.tick_params(labelsize=6.5, direction='in', length=2, width=0.8, pad=1)
    colorbar.set_label('Power (dB)', fontsize=7)

    fig.subplots_adjust(left=0.11, right=0.985, bottom=0.28, top=0.8)

    if save:
        os.makedirs(output_dir, exist_ok=True)
        name = '_'.join([row['site'], row['node'], row['run'].replace('/', '_'), row['band']]) + '.png'
        out = os.path.join(output_dir, name)
        fig.savefig(out, dpi=300, bbox_inches='tight', pad_inches=0.02)
        print('saved', out)
    plt.show()
    return fig


## Plot Selected Bands

This reads each CSV from object storage when it plots. It does not download raw `.npz` files.


In [ ]:
for _, row in selected.iterrows():
    print(row['site'], row['node'], row['run'], row['band'])
    plot_band(row, max_time=MAX_TIME, from_end=FROM_END, save=False)


## Save Figures

Set `save=True` to write PNGs into the Colab runtime.


In [ ]:
# for _, row in selected.iterrows():
#     plot_band(row, max_time=MAX_TIME, from_end=FROM_END, save=True, output_dir='spectrograms')
